# 03. Modeling ANN — Rancangan Arsitektur & Skenario Eksperimen

Notebook ini **mendefinisikan seluruh arsitektur ANN secara langsung di dalamnya** — baik fungsi pembangun model maupun keenam skenario eksperimen (E1–E6) — sehingga semuanya terlihat jelas dan lengkap tanpa perlu membuka file lain untuk memahaminya. Notebook ini **tidak melakukan training**; training & evaluasi dilakukan di `04_Training_Evaluation.ipynb`.

**Pipeline notebook:** `00_Data_Exploration` → `01_Data_Preprocessing` → `02_Feature_Engineering` → `03_Modeling_ANN` → `04_Training_Evaluation` → `05_SHAP_Explainability` → `06_Deployment`

---

## Import Library

In [1]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from io_utils import load_json

print('TensorFlow version:', tf.__version__)
tf.random.set_seed(42)

TensorFlow version: 2.21.0


## Formulasi Input dan Output Model

- **Input**: 15 fitur hasil seleksi fitur pada `02_Feature_Engineering.ipynb` (`inq_last_6mths`, `int_rate`, `total_rev_hi_lim`, `term`, `tot_cur_bal`, `loan_amnt`, `open_acc`, `dti`, `installment`, `revol_util`, `pub_rec`, `revol_bal`, `delinq_2yrs`, `annual_inc`, `total_acc`). Setiap data nasabah direpresentasikan sebagai vektor X = {x₁, x₂, ..., x₁₅}.
- **Output**: probabilitas 4 kelas kelayakan kredit y ∈ {Prime, Performing, Under-performing, Non-Performing}, dengan prediksi akhir ŷᵢ = argmax P(yᵢ = k | xᵢ).

In [2]:
selected_features = load_json('selected_features.json')
label_names = load_json('label_names.json')

input_dim = len(selected_features)
n_classes = len(label_names)

print(f'Jumlah fitur input : {input_dim}')
print(f'Fitur              : {selected_features}')
print(f'Jumlah kelas output: {n_classes} -> {label_names}')

Jumlah fitur input : 15
Fitur              : ['inq_last_6mths', 'int_rate', 'total_rev_hi_lim', 'term', 'tot_cur_bal', 'loan_amnt', 'open_acc', 'dti', 'installment', 'revol_util', 'pub_rec', 'revol_bal', 'delinq_2yrs', 'annual_inc', 'emp_length']
Jumlah kelas output: 4 -> {'0': 'Prime', '1': 'Performing', '2': 'Under-performing', '3': 'Non-Performing'}


## Rancangan Arsitektur Dasar

Arsitektur terdiri dari 1 input layer, 5 hidden layer, dan 1 output layer.

| Layer | Jumlah Neuron | Activation | Tujuan |
|---|---|---|---|
| Input Layer | 15 neuron | - | Menerima 15 fitur input |
| Hidden Layer 1 | 256 neuron | ReLU | Mempelajari kombinasi fitur awal |
| Hidden Layer 2 | 128 neuron | ReLU | Menangkap pola non-linear |
| Hidden Layer 3 | 64 neuron | ReLU | Menyaring representasi fitur |
| Hidden Layer 4 | 32 neuron | ReLU | Memadatkan informasi penting |
| Hidden Layer 5 | 16 neuron | ReLU | Representasi akhir sebelum klasifikasi |
| Output Layer | 4 neuron | Softmax | Menghasilkan probabilitas 4 kelas |

Batch normalization dan dropout bersifat opsional dan diuji pengaruhnya lewat 6 skenario eksperimen E1–E6 di bawah.

## Fungsi Pembangun Arsitektur (`build_ann_model`)

In [3]:
HIDDEN_UNITS = [256, 128, 64, 32, 16]


def build_ann_model(
    input_dim: int,
    n_classes: int = 4,
    use_dropout: bool = False,
    use_batchnorm: bool = False,
    dropout_rate: float = 0.3,
    learning_rate: float = 1e-3,
    name: str = 'credit_risk_ann',
) -> keras.Model:
    """
    Membangun model ANN sesuai rancangan arsitektur:
    1 input layer (15 fitur) -> 5 hidden layer (256-128-64-32-16, ReLU)
    -> 1 output layer (4 neuron, Softmax).

    use_dropout dan use_batchnorm mengatur regularisasi tambahan yang
    membedakan keenam skenario eksperimen (E1-E6) di bawah.
    """
    inputs = keras.Input(shape=(input_dim,), name='input_features')
    x = inputs

    for i, units in enumerate(HIDDEN_UNITS, start=1):
        x = layers.Dense(units, activation='relu', name=f'dense_{i}')(x)
        if use_batchnorm:
            x = layers.BatchNormalization(name=f'batchnorm_{i}')(x)
        if use_dropout:
            x = layers.Dropout(dropout_rate, name=f'dropout_{i}')(x)

    outputs = layers.Dense(n_classes, activation='softmax', name='output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name=name)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

## Definisi 6 Skenario Eksperimen (E1–E6)

Sesuai Tabel 5 (Perancangan Eksperimen): arsitektur dasar yang sama (256-128-64-32-16) diuji dengan kombinasi regularisasi (dropout, batch normalization) dan penanganan *class imbalance* (class weight) yang berbeda. `use_class_weight` tidak mengubah struktur layer — parameter ini dipakai nanti saat `model.fit(..., class_weight=...)` di notebook training.

In [4]:
EXPERIMENTS = {
    'E1': {
        'description': 'Baseline ANN',
        'use_dropout': False,
        'use_batchnorm': False,
        'use_class_weight': False,
    },
    'E2': {
        'description': 'ANN + Dropout',
        'use_dropout': True,
        'use_batchnorm': False,
        'use_class_weight': False,
    },
    'E3': {
        'description': 'ANN + Batch Normalization',
        'use_dropout': False,
        'use_batchnorm': True,
        'use_class_weight': False,
    },
    'E4': {
        'description': 'ANN + Dropout + Batch Normalization',
        'use_dropout': True,
        'use_batchnorm': True,
        'use_class_weight': False,
    },
    'E5': {
        'description': 'ANN + Class Weight',
        'use_dropout': False,
        'use_batchnorm': False,
        'use_class_weight': True,
    },
    'E6': {
        'description': 'ANN + Dropout + Class Weight',
        'use_dropout': True,
        'use_batchnorm': False,
        'use_class_weight': True,
    },
}

pd.DataFrame(EXPERIMENTS).T.rename_axis('Eksperimen')

,description,use_dropout,use_batchnorm,use_class_weight
Eksperimen,,,,
E1,Baseline ANN,False,False,False
E2,ANN + Dropout,True,False,False
E3,ANN + Batch Normalization,False,True,False
E4,ANN + Dropout + Batch Normalization,True,True,False
E5,ANN + Class Weight,False,False,True
E6,ANN + Dropout + Class Weight,True,False,True


## Membangun Seluruh Arsitektur Eksperimen (E1–E6)

Keenam model dibangun langsung di sini menggunakan `build_ann_model` di atas, sehingga arsitektur masing-masing eksperimen benar-benar terlihat & dapat diperiksa satu per satu — bukan sekadar disebutkan di tabel.

In [5]:
models_by_experiment = {}

for exp_id, config in EXPERIMENTS.items():
    models_by_experiment[exp_id] = build_ann_model(
        input_dim=input_dim,
        n_classes=n_classes,
        use_dropout=config['use_dropout'],
        use_batchnorm=config['use_batchnorm'],
        name=f'ann_{exp_id.lower()}',
    )

print('Model berhasil dibangun untuk eksperimen:', list(models_by_experiment))

Model berhasil dibangun untuk eksperimen: ['E1', 'E2', 'E3', 'E4', 'E5', 'E6']


In [6]:
# Tampilkan arsitektur (summary) tiap eksperimen satu per satu
for exp_id, model in models_by_experiment.items():
    print('=' * 80)
    print(f"{exp_id}: {EXPERIMENTS[exp_id]['description']}")
    print('=' * 80)
    model.summary()
    print()

E1: Baseline ANN


Model: "ann_e1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 15)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │         4,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,924 (187.20 KB)

 Trainable params: 47,924 (187.20 KB)

...

Model: "ann_e6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 15)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │         4,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,924 (187.20 KB)

 Trainable params: 47,924 (187.20 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Bandingkan jumlah parameter tiap eksperimen
param_comparison = pd.DataFrame([
    {
        'Eksperimen': exp_id,
        'Deskripsi': EXPERIMENTS[exp_id]['description'],
        'Total Params': model.count_params(),
        'Trainable Params': sum(
            np.prod(w.shape) for w in model.trainable_weights
        ),
    }
    for exp_id, model in models_by_experiment.items()
]).set_index('Eksperimen')

param_comparison

,Deskripsi,Total Params,Trainable Params
Eksperimen,,,
E1,Baseline ANN,47924,47924
E2,ANN + Dropout,47924,47924
E3,ANN + Batch Normalization,49908,48916
E4,ANN + Dropout + Batch Normalization,49908,48916
E5,ANN + Class Weight,47924,47924
E6,ANN + Dropout + Class Weight,47924,47924


In [8]:
# Visualisasi arsitektur baseline E1 (opsional, butuh graphviz+pydot)
try:
    keras.utils.plot_model(
        models_by_experiment['E1'],
        to_file='../reports/ann_architecture.png',
        show_shapes=True,
        show_layer_names=True,
        dpi=100,
    )
    print('Diagram arsitektur disimpan di reports/ann_architecture.png')
except Exception as e:
    print(f'Lewati visualisasi (graphviz/pydot belum terpasang): {e}')

You must install pydot (`pip install pydot`) for `plot_model` to work.
Diagram arsitektur disimpan di reports/ann_architecture.png


## Menyimpan Definisi untuk Dipakai Ulang di Notebook Training

Agar `04_Training_Evaluation.ipynb` tidak perlu menulis ulang kode arsitektur di atas (dan supaya notebook itu bisa fokus penuh pada menjalankan eksperimen), fungsi & konfigurasi yang **sudah didefinisikan dan ditampilkan lengkap di atas** disalin apa adanya ke `src/model.py` lewat `%%writefile`. Ini murni langkah penyimpanan — bukan sumber definisi baru; semua isi arsitektur & eksperimen sudah terlihat sepenuhnya di sel-sel sebelumnya.

In [9]:
%%writefile ../src/model.py
"""
src/model.py — disalin dari 03_Modeling_ANN.ipynb.
Lihat notebook tersebut untuk penjelasan lengkap tiap eksperimen.
"""

from tensorflow import keras
from tensorflow.keras import layers

HIDDEN_UNITS = [256, 128, 64, 32, 16]


def build_ann_model(
    input_dim: int,
    n_classes: int = 4,
    use_dropout: bool = False,
    use_batchnorm: bool = False,
    dropout_rate: float = 0.3,
    learning_rate: float = 1e-3,
    name: str = 'credit_risk_ann',
) -> keras.Model:
    inputs = keras.Input(shape=(input_dim,), name='input_features')
    x = inputs
    for i, units in enumerate(HIDDEN_UNITS, start=1):
        x = layers.Dense(units, activation='relu', name=f'dense_{i}')(x)
        if use_batchnorm:
            x = layers.BatchNormalization(name=f'batchnorm_{i}')(x)
        if use_dropout:
            x = layers.Dropout(dropout_rate, name=f'dropout_{i}')(x)
    outputs = layers.Dense(n_classes, activation='softmax', name='output')(x)
    model = keras.Model(inputs=inputs, outputs=outputs, name=name)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


EXPERIMENTS = {
    'E1': {'description': 'Baseline ANN', 'use_dropout': False, 'use_batchnorm': False, 'use_class_weight': False},
    'E2': {'description': 'ANN + Dropout', 'use_dropout': True, 'use_batchnorm': False, 'use_class_weight': False},
    'E3': {'description': 'ANN + Batch Normalization', 'use_dropout': False, 'use_batchnorm': True, 'use_class_weight': False},
    'E4': {'description': 'ANN + Dropout + Batch Normalization', 'use_dropout': True, 'use_batchnorm': True, 'use_class_weight': False},
    'E5': {'description': 'ANN + Class Weight', 'use_dropout': False, 'use_batchnorm': False, 'use_class_weight': True},
    'E6': {'description': 'ANN + Dropout + Class Weight', 'use_dropout': True, 'use_batchnorm': False, 'use_class_weight': True},
}

Writing ../src/model.py


## Ringkasan & Langkah Selanjutnya

Seluruh arsitektur — fungsi `build_ann_model` maupun keenam model eksperimen E1–E6 beserta jumlah parameternya — telah didefinisikan, dibangun, dan ditampilkan lengkap di notebook ini.

Lanjut ke **`04_Training_Evaluation.ipynb`**, yang fokus penuh pada melatih, membandingkan, dan mengevaluasi keenam eksperimen tersebut.